# Le maillon faible : la vitesse · *The weak link: velocity*

Notebook compagnon du chapitre **26. La théorie quantitative de la monnaie (MV = PQ) : relier monnaie, prix et activité** — [lire l'article](https://nmlab.io/ressources/theorie-quantitative-monnaie-mv-pq).
Companion notebook to chapter **26. The Quantity Theory of Money (MV = PQ): Linking Money, Prices and Activity** — [read the article](https://nmlab.io/en/ressources/quantity-theory-of-money).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


# (les séries sont chargées dans build_figure)


import numpy as np
import pandas as pd
from matplotlib.figure import Figure
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

C, W = nm.COLORS, nm.WIDTH_PX

EA_M3_KEY = "BSI/M.U2.Y.V.M30.X.1.U2.2300.Z01.E"   # encours M3 zone euro (BCE)


def load(series_id: str, start: str | None = None, end: str | None = None) -> pd.Series:
    """Charge une série en direct : FRED, ou le portail de la BCE pour « EA_M3 »."""
    if series_id == "EA_M3":
        url = (f"https://data-api.ecb.europa.eu/service/data/{EA_M3_KEY}"
               "?format=csvdata&detail=dataonly")
        raw = pd.read_csv(url)
        s = pd.Series(raw["OBS_VALUE"].values,
                      index=pd.PeriodIndex(raw["TIME_PERIOD"], freq="M").to_timestamp())
        s = s.sort_index() / 1000.0                 # millions -> milliards d'euros
    else:
        s = nm.load_fred(series_id)
    return s.loc[start:end]


def T(d: dict, lang: str):
    """Sélectionne le jeu de libellés de la langue demandée."""
    return d[lang]


def build_figure(lang: str = "fr") -> Figure:
    """Construit la figure NMLab du chapitre (libellés selon ``lang``)."""
    v=load("M2V","1959-01")
    fig=nm.figure(1010); ax=nm.axes(fig)
    ax.plot(v.index,v.values,color=C["teal"],lw=3.2)
    d=dict(fr=("Le maillon faible : la vitesse","Vitesse de circulation de M2 aux États-Unis (PIB nominal ÷ M2).",
               "sommet 2,19 (1997)","plancher 1,13 (2020)","Loin d'être stable, la vitesse a été divisée par près de deux depuis 1997.\nUne chute de V annule une hausse de M. Source : FRED (M2V)."),
           en=("The weak link: velocity","Velocity of M2 in the United States (nominal GDP ÷ M2).",
               "peak 2.19 (1997)","trough 1.13 (2020)","Far from stable, velocity nearly halved since 1997.\nA fall in V cancels a rise in M. Source: FRED (M2V)."))
    t=T(d,lang); nm.header(fig,t[0],t[1])
    xpk=v.idxmax(); xtr=v["2019":].idxmin()
    ax.scatter([xpk],[v.max()],s=120,color=C["amber"],zorder=4)
    ax.scatter([xtr],[v[xtr]],s=120,color=C["rose"],zorder=4)
    ax.text(xpk,v.max()+0.07,t[2],color=C["amber"],fontsize=17,ha="center",va="bottom",fontweight="bold")
    ax.text(xtr,v[xtr]-0.08,t[3],color=C["rose"],fontsize=17,ha="center",va="top",fontweight="bold")
    ax.set_ylim(1.0,2.4)
    nm.footer(fig,t[4]);
    return fig


fig = build_figure(LANG)